# The DS Excursion

CS 301 — Introduction to Data Science, NJIT. Meeting 3: *Confidence Instead of Certainty*.

This notebook loads one real dataset and looks at it, one step at a time.

**The data.** Newark Liberty International Airport, one row per day, 2006 through 2024, from NOAA's GHCN-Daily record (station USW00014734). Each row holds the day's high and low temperature, its average humidity and sea-level pressure, and a few other readings.

**The label.** A label is the answer attached to a row — the thing a classifier is asked to produce from the row's other numbers. Here each day is labelled 1 if it falls in the warm half of the year (May through October) and 0 if it falls in the cold half (November through April).

**The inputs.** The four numbers a day is judged from — its features, the numbers a classifier is given to work with — are the high, the low, the humidity and the pressure. The label is computed from the date, so the date is never an input: anything read off the date (the month, the day of the year) would give the answer away.

The question is: from those four numbers, which half of the year is it?

Run each cell in order, top to bottom, with Shift+Enter.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL = "https://ikoutis.github.io/course-notes/artificial-neuron-notes/data/newark-weather.csv"
df = pd.read_csv(URL, parse_dates=["date"])
df.head()

## The columns

`df` is a DataFrame: a table held in memory, with one row per record and one named column per quantity. Here a row is a day and a column is a measurement. `df.head()` shows the first five rows.

- `date` — the calendar day (not an input; see above)
- `tmax_f` — daily high temperature, °F
- `tmin_f` — daily low temperature, °F
- `humidity_pct` — average relative humidity, percent
- `pressure_hpa` — average sea-level pressure, hectopascals
- `precip_in` — precipitation, inches
- `wind_mph` — average wind speed, miles per hour
- `warm_half` — the label: 1 for May–October, 0 for November–April

`df.shape` is the pair (rows, columns). `df.describe()` gives, for each column, how many values are present, their mean and standard deviation, their smallest and largest values, and the three quartiles. For the `date` column the smallest and largest values are the first and last day in the file.

In [ ]:
print(df.shape)
df.describe().round(1)

## Holes

A truth table has no missing rows. A weather station has days on which an instrument did not report, and the file keeps those as empty fields. `df.isna().sum()` counts the empty fields in each column.

The cell below drops every day that is missing any of the four features and keeps the rest. The label is never missing, because it comes from the date.

In [ ]:
print(df.isna().sum())
FEATURES = ["tmax_f", "tmin_f", "humidity_pct", "pressure_hpa"]
df = df.dropna(subset=FEATURES).reset_index(drop=True)
len(df)

## The label

`value_counts()` counts how many days carry each label. The two classes are nearly the same size — about half the days carry each label. That matters for reading any accuracy later on: a rule that said "warm half" on every day would already be right about half the time, so a rule worth having must do clearly better than that.

In [ ]:
counts = df["warm_half"].value_counts().sort_index()
print(counts)
fig, ax = plt.subplots(figsize=(4, 3))
ax.bar(["cold half (0)", "warm half (1)"], counts.to_numpy(), color=["#1A1A1A", "#CC0033"])
ax.set_ylabel("days")
plt.show()

## One measurement

Take the high alone. The histogram below counts the days at each high, in 5°F bins, drawn separately for the two classes and laid over each other.

Below about 55°F almost every day is cold half; above about 75°F almost every day is warm half. In between, the two shapes overlap: a day with a high of 65°F is warm half about as often as not. On the high alone, those days cannot be told apart.

In [ ]:
bins = np.arange(10, 115, 5)
warm = df[df["warm_half"] == 1]
cold = df[df["warm_half"] == 0]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(warm["tmax_f"], bins=bins, color="#CC0033", alpha=0.55, label="warm half (May–Oct)")
ax.hist(cold["tmax_f"], bins=bins, color="#1A1A1A", alpha=0.35, label="cold half (Nov–Apr)")
ax.set_xlabel("daily high (°F)"); ax.set_ylabel("days"); ax.legend()
plt.show()

## Two measurements

Now the high and the low together. Each day is one point — high on the horizontal axis, low on the vertical — coloured by its label.

A truth table with two inputs has four points. This plot has 6,865. The two clouds are well separated at the ends and overlap along a band in the middle, and no point in the band comes with its answer attached.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(warm["tmax_f"], warm["tmin_f"], s=5, color="#CC0033", alpha=0.45, label="warm half (May–Oct)")
ax.scatter(cold["tmax_f"], cold["tmin_f"], s=5, color="#1A1A1A", alpha=0.30, label="cold half (Nov–Apr)")
ax.set_xlabel("daily high (°F)"); ax.set_ylabel("daily low (°F)"); ax.legend()
plt.show()

## Surprising days

Some points sit deep inside the other cloud. The cell below lists the November days with a high of at least 80°F, and then the May days with a high of at most 55°F. `df["date"].dt.month` reads the month out of each date; the month is used here only to look days up, never as an input.

In [ ]:
month = df["date"].dt.month
cols = ["date", "tmax_f", "tmin_f", "humidity_pct", "pressure_hpa", "warm_half"]
print(df.loc[(month == 11) & (df["tmax_f"] >= 80), cols].to_string(index=False))
print()
print(df.loc[(month == 5) & (df["tmax_f"] <= 55), cols].to_string(index=False))

Every one of these labels is correct. On 2024-11-06 the high was 83°F, and it was November; on 2020-05-09 the high was 50°F, and it was May. The label records which half of the year a day belongs to, not what the weather was like. No rule on the four measurements can put 2024-11-06 on the cold side without putting a typical June day there too.

## A line by hand

Meeting 2 drew a line across the square of a two-input gate and called one side "fires". The same can be done on the (high, low) plane. Here is one such line, chosen by hand: **high + low ≥ 120**, that is, the average of the high and the low is at least 60°F.

The score of a day is the number high + low − 120: positive on the warm side of the line, negative on the cold side, zero on the line itself. The guess is 1 where the score is at least 0 and 0 elsewhere. Accuracy is the fraction of days on which the guess equals the label.

In [ ]:
score = df["tmax_f"] + df["tmin_f"] - 120
guess = (score >= 0).astype(int)
right = guess == df["warm_half"]
print(f"accuracy: {right.mean():.3f}")
print(f"wrong: {(~right).sum()} of {len(df)} days")

In [ ]:
wrong = df[~right]
xs = np.array([30, 110])
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(warm["tmax_f"], warm["tmin_f"], s=5, color="#CC0033", alpha=0.45, label="warm half")
ax.scatter(cold["tmax_f"], cold["tmin_f"], s=5, color="#1A1A1A", alpha=0.30, label="cold half")
ax.scatter(wrong["tmax_f"], wrong["tmin_f"], s=8, color="#F5A623", label="wrong side of the line")
ax.plot(xs, 120 - xs, color="#CC0033", lw=1.2, label="high + low = 120")
ax.set_xlabel("daily high (°F)"); ax.set_ylabel("daily low (°F)"); ax.legend()
plt.show()

## The unit's confidence

A score by itself is unbounded: on this data it runs from about −100 to about +75. The sigmoid is the function σ(s) = 1 / (1 + e^(−s)), which turns any score into a number strictly between 0 and 1, with σ(0) = 1/2; its value is read as the unit's confidence that the label is 1.

Dividing the score by a number T before applying the sigmoid sets how far from the line a day has to be before the unit is nearly sure. The score here is in °F, so T is a width in °F. The table below uses T = 10.

For 2024-11-06 the score is 82.9 + 63.0 − 120 = 25.9, and σ(25.9 / 10) = 0.93. The unit says "93% warm half". The day is cold half. The unit was wrong — and it gave 7% to what happened. A unit that answers only 0 or 1 gave it 0%.

In [ ]:
def sigmoid(s, T=1.0):
    return 1 / (1 + np.exp(-s / T))

days = pd.to_datetime(["2022-11-07", "2024-11-01", "2024-11-06", "2020-05-09"])
table = df[df["date"].isin(days)][["date", "tmax_f", "tmin_f", "warm_half"]].reset_index(drop=True)
table["score"] = table["tmax_f"] + table["tmin_f"] - 120
table["confidence (T=10)"] = sigmoid(table["score"], T=10).round(2)
table

In [ ]:
s = np.linspace(-60, 60, 400)
fig, ax = plt.subplots(figsize=(6, 3.5))
for T, colour in [(4, "#CC0033"), (10, "#1A1A1A"), (20, "#F5A623")]:
    ax.plot(s, sigmoid(s, T), color=colour, label=f"T = {T}")
ax.axhline(0.5, color="#888888", lw=0.8, ls=":")
ax.axvline(0, color="#888888", lw=0.8, ls=":")
ax.set_xlabel("score = high + low − 120 (°F)"); ax.set_ylabel("σ(score / T)"); ax.legend()
plt.show()

## Try it

1. **Move the line.** Change the 120 in the score to another number, or weight the high and the low differently (for example 2 × high + low − 180), and rerun the accuracy cell. Can you get fewer than 707 days wrong?
2. **Change T.** Rerun the table with T = 4 and with T = 20. Which days' confidences move the most, and which barely move?
3. **The high alone.** For a threshold t, the rule "warm half if the high is at least t" has an accuracy. Find the t that does best.

---

**Data.** NOAA National Centers for Environmental Information, Global Historical Climatology Network – Daily (GHCN-Daily), station USW00014734, Newark Liberty International Airport, NJ; the per-station file at `https://www.ncei.noaa.gov/data/global-historical-climatology-network-daily/access/USW00014734.csv`, days 2006-01-01 through 2024-12-31. The course's copy converts NOAA's storage units to °F, hectopascals, inches and miles per hour and adds the `warm_half` label; nothing else is changed, and missing readings are left missing.

The unit assignment loads this same file.